# 03 — How forecastable are USDA livestock prices?

The 6-month forecast bake-off, run on two anchor series:

* **`cattle_steer_choice_nebraska`** — the Choice 65-80% Nebraska direct
  fed-cattle benchmark
* **`boxed_beef_cutout_choice`** — the Choice 1-3 600-900 lb boxed beef
  cutout (the wholesale anchor)

Three forecasters compete: **AutoARIMA**, **Prophet**, and a
**LightGBM** with lag/rolling/calendar features. Each is evaluated with
12-window rolling-origin cross-validation at a 6-month horizon, then the
best model produces a forward 12-month forecast with 80% prediction
intervals.

Run order: top to bottom. Expect ~2-3 minutes of compute on a laptop
(72 model fits in total). Assumes `data/clean/observations.parquet` exists.


In [1]:
from __future__ import annotations

import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import polars as pl
import scipy.stats as stats
from plotly.subplots import make_subplots

from usda_sandbox.forecast import (
    LightGBMForecaster,
    ProphetForecaster,
    StatsForecastAutoARIMA,
    run_backtest,
)
from usda_sandbox.store import read_series


def find_project_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find project root")


PROJECT_ROOT = find_project_root()
OBS_PATH = PROJECT_ROOT / "data" / "clean" / "observations.parquet"

ANCHORS: list[tuple[str, str]] = [
    ("cattle_steer_choice_nebraska", "NE Steers (Choice 65-80%)"),
    ("boxed_beef_cutout_choice", "Boxed beef cutout — Choice"),
]
HORIZON = 6
N_WINDOWS = 12

FORECASTER_REGISTRY = {
    "AutoARIMA": StatsForecastAutoARIMA,
    "Prophet": ProphetForecaster,
    "LightGBM": LightGBMForecaster,
}


## 1. Run the backtests

Twelve rolling-origin windows × six-month horizon × three models per anchor
series. The cell prints how long each backtest took.

In [2]:
results = {}
for series_id, label in ANCHORS:
    t0 = time.time()
    results[series_id] = run_backtest(
        series_id,
        horizon=HORIZON,
        n_windows=N_WINDOWS,
        obs_path=OBS_PATH,
    )
    print(f"{label:<35} done in {time.time() - t0:5.1f}s "
          f"({N_WINDOWS} windows x {HORIZON} months x 3 models)")


10:22:58 - cmdstanpy - INFO - Chain [1] start processing


10:22:58 - cmdstanpy - INFO - Chain [1] done processing


10:22:58 - cmdstanpy - INFO - Chain [1] start processing


10:22:58 - cmdstanpy - INFO - Chain [1] done processing


10:22:58 - cmdstanpy - INFO - Chain [1] start processing


...

10:23:43 - cmdstanpy - INFO - Chain [1] start processing


10:23:43 - cmdstanpy - INFO - Chain [1] done processing


10:23:43 - cmdstanpy - INFO - Chain [1] start processing


10:23:43 - cmdstanpy - INFO - Chain [1] done processing


Boxed beef cutout — Choice          done in  41.0s (12 windows x 6 months x 3 models)


## 2. The scoreboard

Per-model accuracy across all 72 forecast points (12 windows × 6 months) for
each anchor. Lower is better on every metric.

* **MAPE** — mean absolute percent error; the natural framing for prices
* **sMAPE** — symmetric MAPE; less sensitive to small denominators
* **MASE** — mean absolute scaled error; how the model compares to a naive
  one-step forecast (MASE < 1 means the model beats naive; MASE > 1 means
  it loses to it)

In [3]:
combined_metrics = pl.concat(
    [
        results[sid].metrics.with_columns(
            anchor=pl.lit(label),
            series_id=pl.lit(sid),
        )
        for sid, label in ANCHORS
    ]
).select(["anchor", "model", "n_obs", "mape", "smape", "mase"]).sort(["anchor", "mape"])

combined_metrics


anchor,model,n_obs,mape,smape,mase
str,str,i64,f64,f64,f64
"""Boxed beef cutout — Choice""","""Prophet""",72,8.075615,8.814802,3.058099
"""Boxed beef cutout — Choice""","""LightGBM""",72,8.558203,8.854316,3.119182
"""Boxed beef cutout — Choice""","""AutoARIMA""",72,8.571378,8.993514,3.097949
"""NE Steers (Choice 65-80%)""","""AutoARIMA""",71,6.294923,6.463694,2.93562
"""NE Steers (Choice 65-80%)""","""LightGBM""",71,7.731299,7.913743,3.515123
"""NE Steers (Choice 65-80%)""","""Prophet""",71,14.720551,16.391895,6.718701


### 2a. Pick the winner per anchor

In [4]:
winners: dict[str, str] = {}
for sid, label in ANCHORS:
    metrics = results[sid].metrics.sort("mape")
    best = metrics["model"][0]
    winners[sid] = best
    print(
        f"{label:<35} winner: {best:<10}"
        f" (MAPE {metrics['mape'][0]:.2f}, sMAPE {metrics['smape'][0]:.2f},"
        f" MASE {metrics['mase'][0]:.2f})"
    )


NE Steers (Choice 65-80%)           winner: AutoARIMA  (MAPE 6.29, sMAPE 6.46, MASE 2.94)
Boxed beef cutout — Choice          winner: Prophet    (MAPE 8.08, sMAPE 8.81, MASE 3.06)


## 3. Actuals vs. forecasts across all CV windows

Each chart shows the full history (thin gray) and every model's CV-window
forecasts (colored segments, one segment per window). Reading from left to
right walks forward through time and through the 12 evaluation windows.

In [5]:
def plot_cv_overlay(series_id: str, label: str) -> None:
    history = (
        read_series(series_id, OBS_PATH)
        .filter(pl.col("value").is_not_null())
        .sort("period_start")
    )
    cv = results[series_id].cv_details
    cv_min = cv["period_start"].min()
    history_recent = history.filter(pl.col("period_start") >= cv_min - pd.Timedelta(days=365 * 2))

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=history_recent["period_start"].to_list(),
            y=history_recent["value"].to_list(),
            mode="lines",
            name="Actual",
            line=dict(color="rgba(50,50,50,0.85)", width=1.4),
        )
    )

    palette = {"AutoARIMA": "#1f77b4", "Prophet": "#2ca02c", "LightGBM": "#d62728"}
    for model_name, color in palette.items():
        sub = cv.filter(pl.col("model") == model_name).sort(["window", "period_start"])
        # One trace per window so the segments break between windows
        for w in sorted(sub["window"].unique().to_list()):
            wdf = sub.filter(pl.col("window") == w)
            fig.add_trace(
                go.Scatter(
                    x=wdf["period_start"].to_list(),
                    y=wdf["point"].to_list(),
                    mode="lines",
                    line=dict(color=color, width=1.4),
                    name=model_name,
                    legendgroup=model_name,
                    showlegend=(w == sub["window"].min()),
                )
            )

    fig.update_layout(
        title=f"{label} — actuals vs. CV forecasts (h={HORIZON}, {N_WINDOWS} windows)",
        xaxis_title="Period",
        yaxis_title="USD/cwt",
        height=480,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=-0.25),
    )
    fig.show()


for sid, label in ANCHORS:
    plot_cv_overlay(sid, label)


## 4. Out-of-sample residual diagnostics for the per-anchor winner

These are residuals from the cross-validation predictions (not in-sample
fitted values), which is a stricter test: they show how the model fails on
forecasts it actually had to make. Three views per anchor:

1. **Distribution** — should be roughly centered on zero. Big tails or a
   shifted center mean systematic bias
2. **By forecast horizon** — residual size by step-ahead. If it grows
   monotonically, errors compound predictably; if it doesn't, the model is
   doing roughly as well at month 6 as at month 1
3. **Q-Q vs normal** — heavy-tailed residuals tell you the 80% PI is too
   tight (intervals miss reality more often than 20% of the time)

In [6]:
def plot_residual_diagnostics(series_id: str, label: str) -> None:
    winner = winners[series_id]
    cv = (
        results[series_id].cv_details
        .filter(pl.col("model") == winner)
        .sort(["window", "period_start"])
        .with_columns(
            residual=pl.col("actual") - pl.col("point"),
            step=pl.col("period_start").rank(method="ordinal").over("window"),
        )
    )

    residuals = np.asarray(cv["residual"].to_list(), dtype=float)

    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=(
            f"Residual distribution (mean={residuals.mean():.2f}, sd={residuals.std():.2f})",
            "Residual size vs. forecast horizon",
            "Q-Q vs. standard normal",
        ),
    )

    fig.add_trace(
        go.Histogram(x=residuals, nbinsx=20, marker_color="rgba(31,119,180,0.7)"),
        row=1,
        col=1,
    )
    fig.add_vline(x=0, line_color="black", line_width=1, row=1, col=1)

    by_step = (
        cv.group_by("step")
          .agg(pl.col("residual").abs().mean().alias("mae"))
          .sort("step")
    )
    fig.add_trace(
        go.Bar(
            x=by_step["step"].to_list(),
            y=by_step["mae"].to_list(),
            marker_color="rgba(214,39,40,0.75)",
            showlegend=False,
        ),
        row=1,
        col=2,
    )
    fig.update_xaxes(title_text="Months ahead", row=1, col=2)
    fig.update_yaxes(title_text="Mean abs error", row=1, col=2)

    osm, osr = stats.probplot(residuals, dist="norm", fit=False)
    fig.add_trace(
        go.Scatter(
            x=osm,
            y=osr,
            mode="markers",
            marker=dict(color="rgba(44,160,44,0.8)", size=5),
            showlegend=False,
        ),
        row=1,
        col=3,
    )
    # Reference y = x*sigma + mu line
    line_x = np.array([osm.min(), osm.max()])
    line_y = line_x * residuals.std() + residuals.mean()
    fig.add_trace(
        go.Scatter(
            x=line_x,
            y=line_y,
            mode="lines",
            line=dict(color="black", dash="dot", width=1),
            showlegend=False,
        ),
        row=1,
        col=3,
    )
    fig.update_xaxes(title_text="Theoretical quantile", row=1, col=3)
    fig.update_yaxes(title_text="Sample quantile", row=1, col=3)

    fig.update_layout(
        title_text=f"{label} — out-of-sample residual diagnostics ({winner})",
        height=380,
        showlegend=False,
    )
    fig.show()


for sid, label in ANCHORS:
    plot_residual_diagnostics(sid, label)


## 5. Forward 12-month forecast with 80% prediction intervals

Refit the winning model on the **full** series and project 12 months ahead.
This is the headline chart: actuals to date plus where the winning model
thinks the next year is going, with the 80% PI shaded.

In [7]:
def plot_forward_forecast(series_id: str, label: str) -> None:
    history = (
        read_series(series_id, OBS_PATH)
        .filter(pl.col("value").is_not_null())
        .sort("period_start")
    )
    winner = winners[series_id]
    forecaster = FORECASTER_REGISTRY[winner](seed=42)
    forecaster.fit(history)
    forward = forecaster.predict(horizon=12)

    # Tail of history for context
    tail = history.tail(60)

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=tail["period_start"].to_list(),
            y=tail["value"].to_list(),
            mode="lines",
            name="Historical",
            line=dict(color="rgba(50,50,50,0.9)", width=1.6),
        )
    )

    # 80% PI as a filled band
    upper_x = forward["period_start"].to_list()
    upper_y = forward["upper_80"].to_list()
    lower_x = list(reversed(upper_x))
    lower_y = list(reversed(forward["lower_80"].to_list()))
    fig.add_trace(
        go.Scatter(
            x=upper_x + lower_x,
            y=upper_y + lower_y,
            fill="toself",
            fillcolor="rgba(31,119,180,0.18)",
            line=dict(color="rgba(0,0,0,0)"),
            name="80% PI",
            hoverinfo="skip",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=forward["period_start"].to_list(),
            y=forward["point"].to_list(),
            mode="lines+markers",
            name=f"{winner} forecast",
            line=dict(color="rgb(31,119,180)", width=2),
            marker=dict(size=6),
        )
    )

    # Vertical line at the boundary between history and forecast
    last_actual = history["period_start"].max()
    fig.add_vline(x=last_actual, line_color="rgba(120,120,120,0.6)", line_dash="dot")

    fig.update_layout(
        title=f"{label} — {winner} 12-month forecast",
        xaxis_title="Period",
        yaxis_title="USD/cwt",
        height=460,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=-0.25),
    )
    fig.show()

    print(f"\nNumeric forecast for {label} ({winner}):")
    print(forward.with_columns(pl.col("point").round(2),
                               pl.col("lower_80").round(2),
                               pl.col("upper_80").round(2)))


for sid, label in ANCHORS:
    plot_forward_forecast(sid, label)


10:23:52 - cmdstanpy - INFO - Chain [1] start processing



Numeric forecast for NE Steers (Choice 65-80%) (AutoARIMA):
shape: (12, 4)
┌──────────────┬────────┬──────────┬──────────┐
│ period_start ┆ point  ┆ lower_80 ┆ upper_80 │
│ ---          ┆ ---    ┆ ---      ┆ ---      │
│ date         ┆ f64    ┆ f64      ┆ f64      │
╞══════════════╪════════╪══════════╪══════════╡
│ 2026-04-01   ┆ 233.57 ┆ 227.76   ┆ 239.38   │
│ 2026-05-01   ┆ 234.38 ┆ 225.09   ┆ 243.66   │
│ 2026-06-01   ┆ 238.53 ┆ 226.65   ┆ 250.41   │
│ 2026-07-01   ┆ 240.95 ┆ 227.53   ┆ 254.37   │
│ 2026-08-01   ┆ 243.37 ┆ 228.8    ┆ 257.93   │
│ …            ┆ …      ┆ …        ┆ …        │
│ 2026-11-01   ┆ 236.62 ┆ 218.99   ┆ 254.24   │
│ 2026-12-01   ┆ 239.67 ┆ 221.06   ┆ 258.28   │
│ 2027-01-01   ┆ 242.78 ┆ 223.25   ┆ 262.32   │
│ 2027-02-01   ┆ 245.75 ┆ 225.34   ┆ 266.16   │
│ 2027-03-01   ┆ 244.24 ┆ 223.0    ┆ 265.48   │
└──────────────┴────────┴──────────┴──────────┘


10:23:52 - cmdstanpy - INFO - Chain [1] done processing



Numeric forecast for Boxed beef cutout — Choice (Prophet):
shape: (12, 4)
┌──────────────┬────────┬──────────┬──────────┐
│ period_start ┆ point  ┆ lower_80 ┆ upper_80 │
│ ---          ┆ ---    ┆ ---      ┆ ---      │
│ date         ┆ f64    ┆ f64      ┆ f64      │
╞══════════════╪════════╪══════════╪══════════╡
│ 2026-04-01   ┆ 363.56 ┆ 337.37   ┆ 387.89   │
│ 2026-05-01   ┆ 370.67 ┆ 346.34   ┆ 395.66   │
│ 2026-06-01   ┆ 369.09 ┆ 345.61   ┆ 394.03   │
│ 2026-07-01   ┆ 361.43 ┆ 337.05   ┆ 387.98   │
│ 2026-08-01   ┆ 365.36 ┆ 340.98   ┆ 389.77   │
│ …            ┆ …      ┆ …        ┆ …        │
│ 2026-11-01   ┆ 366.65 ┆ 343.12   ┆ 393.1    │
│ 2026-12-01   ┆ 364.36 ┆ 338.5    ┆ 390.45   │
│ 2027-01-01   ┆ 369.38 ┆ 345.02   ┆ 393.86   │
│ 2027-02-01   ┆ 368.78 ┆ 342.63   ┆ 393.34   │
│ 2027-03-01   ┆ 381.57 ┆ 356.75   ┆ 407.01   │
└──────────────┴────────┴──────────┴──────────┘


## Takeaway — is forecasting useful on this data?

**Short answer: yes for direction and range, no for point precision. The
best-per-anchor MAPE is 6-8% at a 6-month horizon, but no model beats a
naive one-step forecast on MASE — and Prophet's variance across anchors is
big enough that "always pick AutoARIMA" is probably the wrong heuristic.**

What the numbers above actually say:

* **No single model wins both anchors.** AutoARIMA dominates the NE Steers
  anchor at MAPE 6.3% (Prophet collapses to 14.7% on the same series — see
  caveat below). On boxed beef cutout, all three models cluster within
  about half a point: Prophet 8.08%, LightGBM 8.56%, AutoARIMA 8.57%.
  Pick the model per series, not project-wide.
* **Prophet is high-variance.** It's the boxed-beef winner but the worst
  fit on Nebraska steers by a wide margin — likely because Prophet's
  changepoint detection over-reacts to the post-2020 cattle-cycle regime
  shift in fed steers. A useful flag: if a series has had a recent,
  durable level-change, AutoARIMA is the safer default.
* **MASE > 1 across the board (2.9 - 6.7).** None of the three models
  beats a naive one-step forecast on absolute scaled error. Translation:
  six months out, model errors are ~3x the typical month-to-month price
  change. That's expected at this horizon — the forecast is still useful
  for "approximate level," not for tactical timing.
* **Residuals are roughly centered, mildly heavy-tailed.** The Q-Q plots
  curve at both ends, meaning the 80% PI is too tight during shock
  periods. Read the interval as "80% under business-as-usual," not as a
  hard probability statement.
* **Errors grow with horizon, modestly.** Month-1-ahead MAE is roughly
  half of month-6-ahead MAE. Forecast usefulness degrades with horizon
  but doesn't collapse — six months is still inside the useful-information
  window.

**Implication for the Tenure pricing module.** A classical forecast can be
the data layer behind a 6-month-ahead price-band feature, framed as
*point + interval* rather than "the price will be X." Don't lead with
LightGBM until exogenous regressors (CME futures, weather, feed costs) are
in scope — on bare price history the GBM doesn't beat the simpler models
enough to justify its complexity.

Next experiments worth running (out of scope for v0.1):

1. **Add CME futures as exogenous regressors** for AutoARIMA (ARIMAX) and
   LightGBM. Futures encode market consensus and should narrow the PI
   meaningfully.
2. **Hybrid: AutoARIMA point forecast + LightGBM residual model** —
   classical for the level, ML for what's left.
3. **Conformal prediction** on the CV residuals to calibrate prediction
   intervals — gives a principled fix for the heavy-tailed PI miscoverage.
4. **Per-series model selection** baked into the pricing module, instead
   of a single project-wide default.
